# Analyse de rentabilité et de marge — pas à pas

Ce notebook reprend ce que fait `python -m profit build`, cellule par cellule.
Chaque étape est visible, modifiable, relançable.

**Données** : AdventureWorks (Microsoft, MIT) — ventes fictives de matériel sportif, 2010-2014.  
**Outil** : DuckDB — base SQL en mémoire, sans serveur.  
**Principe** : tout le calcul est en SQL. Python affiche les résultats.

In [ ]:
import duckdb, pandas as pd, matplotlib.pyplot as plt, numpy as np, re
from pathlib import Path
pd.set_option("display.float_format", lambda x: f"{x:,.1f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

P0_START, P0_END = "2011-12-01", "2012-11-30"
P1_START, P1_END = "2012-12-01", "2013-11-30"
params = dict(P0_START=P0_START, P0_END=P0_END, P1_START=P1_START, P1_END=P1_END)

con = duckdb.connect()
RAW = Path("../data/raw")
if not (RAW / "FactResellerSales.csv").exists():
    raise FileNotFoundError("Lancez : python scripts/download_data.py")
print("DuckDB", duckdb.__version__, "| données :", RAW)

## 1. Chargement des données brutes

On charge les CSV dans DuckDB. Séparateur `|`, pas d'en-tête — le schéma est déclaré explicitement.

In [ ]:
sql = open("../sql/00_staging.sql").read().replace("${RAW_DIR}", str(RAW).replace("\\", "/"))
con.execute(sql)
tables = ["stg_fact_reseller_sales","stg_fact_internet_sales","stg_dim_product",
          "stg_dim_product_subcategory","stg_dim_product_category","stg_dim_date",
          "stg_dim_reseller","stg_dim_sales_territory","stg_dim_promotion"]
print(f"{'Table':<35} {'Lignes':>8}"); print("-"*45)
for t in tables:
    print(f"{t:<35} {con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]:>8,}")

## 2. Modèle en étoile

On transforme le staging en un schéma en étoile :
- `fact_sales` : une ligne = une ligne de commande (grain)
- 7 tables de dimensions : produit, date, territoire, revendeur, client, promotion

**Points clés :** `net_sales` = ventes après remises hors taxes. `cogs` = coût standard. `gross_margin` = marge brute standard. `dim_product` est historisée (SCD2) : on utilise `sku`, jamais `product_key`.

In [ ]:
con.execute(open("../sql/01_model.sql").read())
print("fact_sales — aperçu :")
display(con.execute("SELECT sales_line_id, channel, order_date, sku, quantity, "
    "unit_price, discount_amount, net_sales, cogs, gross_margin, is_below_cost "
    "FROM fact_sales LIMIT 5").fetchdf())
print(f"Total : {con.execute('SELECT COUNT(*) FROM fact_sales').fetchone()[0]:,} lignes")
print(con.execute("SELECT channel, COUNT(*) n, MIN(order_date) debut, MAX(order_date) fin "
    "FROM fact_sales GROUP BY 1 ORDER BY 1").fetchdf().to_string(index=False))

## 3. Contrôles qualité (17 vérifications)

Avant d'analyser, on vérifie la cohérence des données. Chaque contrôle doit retourner **0 violation**.
Exemples : identités comptables (net = brut - remise), clés orphelines, doublons.

In [ ]:
txt = open("../sql/02_quality.sql").read()
blocks = re.split(r"^-- name: (\w+)[^\n]*\n", txt, flags=re.M)[1:]
results = []
for name, body in zip(blocks[::2], blocks[1::2]):
    if not name.startswith("chk_"):
        continue
    sql = "\n".join(l for l in body.splitlines() if not l.strip().startswith("--")).strip().rstrip(";")
    for k, v in params.items():
        sql = sql.replace("${" + k + "}", v)
    viol = float(con.execute(sql).fetchone()[0])
    results.append({"controle": name, "violations": viol, "statut": "OK" if viol == 0 else "ECHEC"})
df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\n{(df.violations==0).sum()}/{len(df)} controles passes")

## 4. KPIs par canal — P0 vs P1

Deux fenêtres de 12 mois alignées sur la dernière date des ventes revendeurs (29/11/2013).
Pourquoi décembre→novembre ? Pour que les deux canaux soient comparables sur la même durée.

In [ ]:
con.execute(f"CREATE OR REPLACE VIEW sales_period AS "
    f"SELECT f.*, CASE WHEN f.order_date BETWEEN DATE '{P0_START}' AND DATE '{P0_END}' THEN 'P0' "
    f"WHEN f.order_date BETWEEN DATE '{P1_START}' AND DATE '{P1_END}' THEN 'P1' END AS period "
    f"FROM fact_sales f WHERE f.order_date BETWEEN DATE '{P0_START}' AND DATE '{P1_END}'")

kpi = con.execute("SELECT period, COALESCE(channel,'Total') canal, "
    "ROUND(SUM(net_sales)/1e6,2) ventes_M, ROUND(SUM(gross_margin)/1e6,2) marge_M, "
    "ROUND(100.0*SUM(gross_margin)/SUM(net_sales),1) taux_marge_pct, "
    "ROUND(100.0*AVG(is_below_cost::INT),1) pct_lignes_a_perte "
    "FROM sales_period GROUP BY ROLLUP(period,channel) "
    "HAVING period IS NOT NULL ORDER BY period, channel NULLS FIRST").fetchdf()
print(kpi.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
chans = ["Internet","Revendeurs"]
colors = {"P0":"#8d99ae","P1":"#1d3557"}
for i, (metric, label, fmt) in enumerate([("marge_M","Marge brute (M$)","{:.2f}"),
                                            ("taux_marge_pct","Taux de marge (%)","{:.1f} %")]):
    ax = axes[i]
    for j, period in enumerate(["P0","P1"]):
        vals = [float(kpi[(kpi.period==period)&(kpi.canal==c)][metric].values[0]) for c in chans]
        x = np.arange(len(chans))
        bars = ax.bar(x+(j-.5)*.38, vals, .38, color=colors[period], label=period)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, v+(0.02 if v>=0 else -0.15),
                    fmt.format(v), ha="center", va="bottom", fontsize=8)
    ax.set_xticks(np.arange(len(chans)), chans); ax.set_title(label)
    ax.axhline(0, color="black", lw=0.8); ax.legend(frameon=False)
plt.tight_layout(); plt.show()
print("Internet : ~41% de marge stable")
print("Revendeurs : +4,1% -> -1,9% : c'est le probleme central")

## 5. Pont de marge Prix/Volume/Mix (PVM)

On décompose la variation de marge en 7 effets :

| Effet | Ce qu'il mesure |
|---|---|
| Volume | Croissance des volumes à mix constant |
| Mix | A-t-on vendu plus de produits à forte marge ? |
| Prix | Evolution des prix catalogue |
| Remises | Evolution des remises accordées |
| Coût | Evolution des coûts standards |
| Nouveaux | Marge des références apparues en P1 |
| Arrêtés | Marge perdue sur les références sorties en P0 |

**Identité vérifiée** : Marge P1 - Marge P0 = somme des 7 effets

In [ ]:
con.execute("CREATE OR REPLACE TABLE pvm_lines AS "
    "WITH agg AS (SELECT sku, channel, period, "
    "  CAST(SUM(quantity) AS DOUBLE) q, CAST(SUM(gross_amount) AS DOUBLE) gross, "
    "  CAST(SUM(discount_amount) AS DOUBLE) disc, CAST(SUM(net_sales) AS DOUBLE) net, "
    "  CAST(SUM(cogs) AS DOUBLE) cogs FROM sales_period GROUP BY 1,2,3), "
    "w AS (SELECT COALESCE(a.sku,b.sku) sku, COALESCE(a.channel,b.channel) channel, "
    "  COALESCE(a.q,0) q0, COALESCE(b.q,0) q1, COALESCE(a.gross,0) gross0, COALESCE(b.gross,0) gross1, "
    "  COALESCE(a.disc,0) disc0, COALESCE(b.disc,0) disc1, COALESCE(a.net,0) net0, COALESCE(b.net,0) net1, "
    "  COALESCE(a.cogs,0) cogs0, COALESCE(b.cogs,0) cogs1 "
    "  FROM (SELECT * FROM agg WHERE period='P0') a "
    "  FULL OUTER JOIN (SELECT * FROM agg WHERE period='P1') b ON a.sku=b.sku AND a.channel=b.channel), "
    "m AS (SELECT *, net0-cogs0 margin0, net1-cogs1 margin1 FROM w), "
    "tot AS (SELECT SUM(q0) q0c, SUM(q1) q1c FROM m WHERE q0>0 AND q1>0) "
    "SELECT m.sku, m.channel, m.q0, m.q1, m.margin0, m.margin1, m.margin1-m.margin0 delta_margin, "
    "  CASE WHEN m.q0>0 AND m.q1>0 THEN (tot.q1c/tot.q0c-1)*m.margin0 ELSE 0 END volume_effect, "
    "  CASE WHEN m.q0>0 AND m.q1>0 THEN m.q1*m.margin0/m.q0-m.margin0*tot.q1c/tot.q0c ELSE 0 END mix_effect, "
    "  CASE WHEN m.q0>0 AND m.q1>0 THEN gross1-m.q1*gross0/m.q0 ELSE 0 END price_effect, "
    "  CASE WHEN m.q0>0 AND m.q1>0 THEN -(disc1-m.q1*disc0/m.q0) ELSE 0 END discount_effect, "
    "  CASE WHEN m.q0>0 AND m.q1>0 THEN -(cogs1-m.q1*cogs0/m.q0) ELSE 0 END cost_effect, "
    "  CASE WHEN m.q0=0 THEN m.margin1 ELSE 0 END new_effect, "
    "  CASE WHEN m.q1=0 THEN -m.margin0 ELSE 0 END discontinued_effect "
    "FROM m CROSS JOIN tot")

bridge = con.execute(
    "SELECT 'Marge P0' etape,0 ord,SUM(margin0) val FROM pvm_lines UNION ALL "
    "SELECT 'Volume',1,SUM(volume_effect) FROM pvm_lines UNION ALL "
    "SELECT 'Mix',2,SUM(mix_effect) FROM pvm_lines UNION ALL "
    "SELECT 'Prix catalogue',3,SUM(price_effect) FROM pvm_lines UNION ALL "
    "SELECT 'Remises',4,SUM(discount_effect) FROM pvm_lines UNION ALL "
    "SELECT 'Cout standard',5,SUM(cost_effect) FROM pvm_lines UNION ALL "
    "SELECT 'Nouveaux produits',6,SUM(new_effect) FROM pvm_lines UNION ALL "
    "SELECT 'Produits arretes',7,SUM(discontinued_effect) FROM pvm_lines UNION ALL "
    "SELECT 'Marge P1',8,SUM(margin1) FROM pvm_lines ORDER BY ord").fetchdf()
bridge["M$"] = (bridge["val"]/1e6).round(2)
print(bridge[["etape","M$"]].to_string(index=False))
m0 = bridge[bridge.ord==0]["val"].values[0]
m1 = bridge[bridge.ord==8]["val"].values[0]
eff = bridge[(bridge.ord>0)&(bridge.ord<8)]["val"].sum()
print(f"\nIdentite : {m0/1e6:.2f} + {eff/1e6:.2f} = {(m0+eff)/1e6:.2f} ~ {m1/1e6:.2f} M$")

In [ ]:
steps = bridge["etape"].tolist(); values = bridge["val"].tolist(); n=len(values)
cum=0; bottoms,heights,colors_bar=[],[],[]
for i,v in enumerate(values):
    if i in (0,n-1):
        bottoms.append(min(0,v)); heights.append(abs(v)); colors_bar.append("#1d3557"); cum=v if i==0 else cum
    else:
        bottoms.append(min(cum,cum+v)); heights.append(abs(v))
        colors_bar.append("#2d6a4f" if v>=0 else "#c1121f"); cum+=v
fig,ax=plt.subplots(figsize=(11,4.5))
ax.bar(range(n),heights,bottom=bottoms,color=colors_bar,width=0.6)
span=max(b+h for b,h in zip(bottoms,heights))-min(bottoms+[0])
for i,(b,h,v) in enumerate(zip(bottoms,heights,values)):
    sign="+" if v>0 and i not in (0,n-1) else ""
    if v>=0 or i in (0,n-1): ax.text(i,b+h+span*.015,f"{sign}{v/1e6:.2f}M$",ha="center",va="bottom",fontsize=8)
    else: ax.text(i,b-span*.015,f"{v/1e6:.2f}M$",ha="center",va="top",fontsize=8)
ax.set_xticks(range(n),steps,rotation=20,ha="right"); ax.axhline(0,color="black",lw=0.8)
ax.set_title("Pont de marge P0 -> P1"); ax.yaxis.set_major_formatter(lambda v,_:f"{v/1e6:.1f}M$")
plt.tight_layout(); plt.show()

## 6. Pareto et classes ABC

On classe les références par marge décroissante :
- **A** : 80% de la marge positive
- **B** : 15% suivants
- **C** : traîne positive
- **D** : marge négative (destructrices de valeur)

In [ ]:
pareto = con.execute(
    "WITH dim AS (SELECT CAST(ProductKey AS INT) pk, ProductAlternateKey sku, "
    "  EnglishProductName pname, c.EnglishProductCategoryName cat "
    "  FROM stg_dim_product p "
    "  LEFT JOIN stg_dim_product_subcategory s ON s.ProductSubcategoryKey=TRY_CAST(p.ProductSubcategoryKey AS INT) "
    "  LEFT JOIN stg_dim_product_category c ON c.ProductCategoryKey=s.ProductCategoryKey), "
    "p AS (SELECT f.sku, MAX(d.pname) pname, MAX(d.cat) cat, "
    "  SUM(f.net_sales) net_sales, SUM(f.gross_margin) gross_margin "
    "  FROM sales_period f JOIN dim d ON d.sku=f.sku WHERE f.period='P1' GROUP BY 1), "
    "r AS (SELECT *, SUM(GREATEST(gross_margin,0)) OVER () total_pos, "
    "  SUM(GREATEST(gross_margin,0)) OVER (ORDER BY gross_margin DESC,sku "
    "    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) cum_before, "
    "  SUM(gross_margin) OVER (ORDER BY gross_margin DESC,sku ROWS UNBOUNDED PRECEDING) cum_margin, "
    "  ROW_NUMBER() OVER (ORDER BY gross_margin DESC,sku) rank_m, COUNT(*) OVER () n_items FROM p) "
    "SELECT sku, pname, cat, ROUND(net_sales) net_sales, ROUND(gross_margin) gross_margin, "
    "  ROUND(100.0*gross_margin/NULLIF(net_sales,0),1) margin_pct, "
    "  rank_m, n_items, ROUND(100.0*rank_m/n_items,1) pct_items, "
    "  ROUND(100.0*cum_margin/SUM(GREATEST(gross_margin,0)) OVER(),1) cum_pct, "
    "  CASE WHEN gross_margin<0 THEN 'D' "
    "       WHEN COALESCE(cum_before,0)<0.80*total_pos THEN 'A' "
    "       WHEN COALESCE(cum_before,0)<0.95*total_pos THEN 'B' ELSE 'C' END abc "
    "FROM r ORDER BY rank_m").fetchdf()
summary=pareto.groupby("abc").agg(nb_ref=("sku","count"),marge=("gross_margin","sum")).reset_index()
summary["marge_M"]=(summary["marge"]/1e6).round(2)
print(summary[["abc","nb_ref","marge_M"]].to_string(index=False))
print("\n10 references les plus deficitaires :")
print(pareto.tail(10)[["pname","cat","gross_margin","margin_pct"]].to_string(index=False))

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.5))
colors_abc={"A":"#2d6a4f","B":"#95d5b2","C":"#8d99ae","D":"#c1121f"}
for cls,g in pareto.groupby("abc"):
    ax.scatter(g["pct_items"],g["cum_pct"],s=14,color=colors_abc[cls],label=f"Classe {cls} ({len(g)} ref.)",zorder=4)
ax.plot(pareto["pct_items"],pareto["cum_pct"],color="black",lw=1,zorder=3)
ax.axhline(100,color="grey",ls="--",lw=1)
ax.set_xlabel("Part des references (%)"); ax.set_ylabel("Marge cumulee / marge positive (%)")
ax.set_title("Pareto de la marge brute (P1)"); ax.legend(frameon=False,fontsize=8)
plt.tight_layout(); plt.show()

## 7. Analyse des remises

3 populations de lignes remisées :
1. La remise **crée** la perte (rentable avant, déficitaire après)
2. La ligne était **déjà à perte** avant remise
3. La ligne **reste rentable** après remise

In [ ]:
disc = con.execute(
    "SELECT CASE WHEN margin_before_discount>=0 AND gross_margin<0 THEN '1-Remise cree la perte' "
    "  WHEN margin_before_discount<0 THEN '2-Deja a perte avant remise' "
    "  ELSE '3-Reste rentable' END population, "
    "COUNT(*) lignes, ROUND(SUM(discount_amount)) remises, ROUND(SUM(gross_margin)) marge_apres "
    "FROM sales_period WHERE period='P1' AND discount_amount>0 GROUP BY 1 ORDER BY 1").fetchdf()
print(disc.to_string(index=False))
pct=disc[disc.population.str.startswith("2")]["remises"].values[0]/disc["remises"].sum()*100
print(f"\n{pct:.0f}% des remises vont a des lignes deja a perte -> la remise aggrave, elle ne cause pas")

## 8. Simulations

Deux leviers :
1. **Hausse de prix** sur les lignes déficitaires — gain selon l'élasticité ?
2. **Arrêt des ventes** à perte — à quelle condition est-ce rentable ?

In [ ]:
res = con.execute(
    "SELECT SUM(net_sales) r, SUM(cogs) c, SUM(gross_margin) m FROM "
    "(SELECT sku,channel,SUM(net_sales) net_sales,SUM(cogs) cogs,SUM(gross_margin) gross_margin "
    " FROM sales_period WHERE period='P1' GROUP BY 1,2 HAVING SUM(gross_margin)<0)").fetchone()
revenue,cogs_val,margin=float(res[0]),float(res[1]),float(res[2])
print(f"Ensemble deficitaire: CA {revenue/1e6:.1f}M$ | Cout {cogs_val/1e6:.1f}M$ | Marge {margin/1e6:.2f}M$")

def gain(rev,cgs,x,e,v=1.0):
    vol=max(0.0,1+e*x); return rev*(vol*(1+x)-1)-v*cgs*(vol-1)

hausses=[0.03,0.05,0.08,0.10]; elasticites=[0.0,-1.0,-2.0,-3.0]
grid=pd.DataFrame({f"e={e:.0f}":[gain(revenue,cogs_val,x,e)/1e6 for x in hausses]
                   for e in elasticites}, index=[f"+{x*100:.0f}%" for x in hausses])
print("\nGain de marge (M$/an) selon hausse de prix et elasticite :")
print(grid.round(2).to_string())
print("\nTous positifs : perdre une vente deficitaire reduit la perte -> hausse de prix robuste")

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
im=axes[0].imshow(grid.values,cmap="Greens",aspect="auto",vmin=0)
axes[0].set_xticks(range(len(elasticites)),[f"e={e:.0f}" for e in elasticites])
axes[0].set_yticks(range(len(hausses)),[f"+{x*100:.0f}%" for x in hausses])
for i in range(len(hausses)):
    for j in range(len(elasticites)):
        axes[0].text(j,i,f"{grid.values[i,j]:.2f}",ha="center",va="center",fontsize=9)
axes[0].grid(False); axes[0].set_title("Gain marge (M$/an)\nHausse prix x Elasticite")

seuil=revenue/cogs_val; v_s=np.linspace(.6,1.,100); delta=(v_s*cogs_val-revenue)/1e6
axes[1].plot(v_s*100,delta,color="#1d3557",lw=2)
axes[1].axhline(0,color="black",lw=0.8); axes[1].axvline(seuil*100,color="#c1121f",ls="--",lw=1.5)
axes[1].fill_between(v_s*100,delta,0,where=(delta>0),alpha=0.15,color="#2d6a4f",label="Rentable")
axes[1].fill_between(v_s*100,delta,0,where=(delta<0),alpha=0.15,color="#c1121f",label="Destructeur")
axes[1].annotate(f"Seuil\n{seuil*100:.1f}%",(seuil*100,0),(15,30),
                 textcoords="offset points",arrowprops=dict(arrowstyle="->"),fontsize=9)
axes[1].set_xlabel("Part du cout evitable (%)"); axes[1].set_ylabel("Variation marge (M$)")
axes[1].set_title("Arreter de vendre a perte"); axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()
print(f"Seuil : {seuil*100:.1f}% du cout doit etre evitable -> tres eleve, hausse prix preferee")

## Synthese

| Constat | Chiffre cle |
|---|---|
| Ventes P1 | 51,3 M$ (+52% vs P0) |
| Marge Internet | +6,25 M$ a 41,4% |
| Marge Revendeurs | **-0,67 M$** a -1,9% |
| Pont PVM | Prix +2,38 M$ absorbe par Cout -2,57 M$ |
| Lignes a perte | 71 couples (ref x canal), -2,09 M$ sur 46% du CA |
| Remises | 90% sur des lignes deja deficitaires — symptome, pas cause |
| Levier robuste | +5% de prix -> +1,19 M$/an, positif quelle que soit l'elasticite |